# Fuente: Idescat

Hay diversas formas de accecer a los datos. La más sofisticada es con su API (estándar [OpenAPI](https://www.openapis.org/)) en `taules/v2`. Esta permite tener acceso completo a un dataset, mientras que usar `indicadors` o la web para descargar un `.csv` nos puede obligar a ceñirnos a un subconjunto.

[Documentación de la API de taules.](https://www.idescat.cat/dev/api/taules/)

In [2]:
import pandas as pd
import requests

from pathlib import Path
from pyjstat import pyjstat
from io import StringIO

PATH_DATA = Path("../data")

Por ejemplo, para el dataset de población:

## Método automático 1 (el equivalente al manual):

In [7]:
url = "https://www.idescat.cat/indicadors/?f=ssv&id=aec&n=15233&t=202500"

response = requests.get(url)
response.raise_for_status()

In [3]:
# Investigamos qué devuelve el servidor
for i, line in enumerate(response.text.splitlines()[:10]):
    print(i, repr(line))

0 "\ufeffPoblació a 1 de gener. Per sexe i grups d'edat. Total. Comarques i Aran, àmbits i províncies"
1 '2025'
2 'Font: Idescat. Estimacions de població.'
3 "Institut d'Estadística de Catalunya"
4 'https://www.idescat.cat/indicadors/?id=aec&n=15233&t=202500'
5 ';0-15;16-24;25-44;45-64;65 i més;Total'
6 'Alt Camp;7119;4898;11283;14352;9486;47138'
7 'Alt Empordà;23137;15461;37679;44592;29175;150044'
8 'Alt Penedès;18279;12463;27228;36384;21744;116098'
9 'Alt Urgell;2757;2004;5287;6604;4825;21477'


Hay que saltarse las primeras 5 líneas.

In [8]:
# Generamos el dataframe saltando esas 5 líneas
    # ¿Es el mismo número siempre? Supondremos que sí.
df = pd.read_csv(
    StringIO(response.content.decode("utf-8-sig")),
    sep=";",
    skiprows=5,
    index_col=0
)

print(df.head())

                 0-15  16-24  25-44  45-64 65 i més   Total
Alt Camp         7119   4898  11283  14352     9486   47138
Alt Empordà     23137  15461  37679  44592    29175  150044
Alt Penedès     18279  12463  27228  36384    21744  116098
Alt Urgell       2757   2004   5287   6604     4825   21477
Alta Ribagorça    510    360    990   1348      843    4051


En este caso, los datos están desagregados por comarca y grupos de edad, pero este dataset también permite desagregar por municipio.

## Método automático 2 (más sofisticado: taules API)

[Documentación](https://www.idescat.cat/dev/api/taules/)

Características clave:

- URI base: `https://api.idescat.cat/taules/v2/{estadística}/{node}/{taula}/{geo}[/data][?{filtres}]`

- Métodos HTTP: GET, POST

- Formato: json (JSON-stat)

- Versión: 2.00 (18/9/2023)

Como detalle: Idescat también tiene un servidor MCP.

Hay unas condiciones de uso. Son las que uno esperaría: reconocer el origen, no modificar ni manipular los datos y no hacer un uso abusivo.

Idescat tiene otras APIs, pero esta parece la más relevante.

In [25]:
# Veamos las diferentes estadísticas
url_base = "https://api.idescat.cat/taules/v2"

response = requests.get(url_base)

for i, line in enumerate(response.text.splitlines()):
    print(i, repr(line))

0 '{'
1 '  "class" : "collection",'
2 '  "href" : "https://api.idescat.cat/taules/v2",'
3 '  "label" : "Estadístiques disponibles",'
4 '  "link" : {'
5 '    "item" : [ {'
6 '      "class" : "collection",'
7 '      "href" : "https://api.idescat.cat/taules/v2/afi",'
8 '      "label" : "Afiliats i afiliacions a la Seguretat Social segons residència padronal de l\'afiliat"'
9 '    }, {'
10 '      "class" : "collection",'
11 '      "href" : "https://api.idescat.cat/taules/v2/censph",'
12 '      "label" : "Cens de població i habitatges"'
13 '    }, {'
14 '      "class" : "collection",'
15 '      "href" : "https://api.idescat.cat/taules/v2/comest",'
16 '      "label" : "Comerç amb l\'estranger"'
17 '    }, {'
18 '      "class" : "collection",'
19 '      "href" : "https://api.idescat.cat/taules/v2/comtec",'
20 '      "label" : "Comerç amb l\'estranger segons contingut tecnològic"'
21 '    }, {'
22 '      "class" : "collection",'
23 '      "href" : "https://api.idescat.cat/taules/v2/pibt",'
24 

In [22]:
url = "https://api.idescat.cat/taules/v2/pmh/446/477/mun/data"

response = requests.get(
    url,
    params={"_LAST_": 5, "lang": "en"}
)

response.raise_for_status()

dataset = pyjstat.Dataset.read(response.text)
df = dataset.write("dataframe")

print(df.head())
print(df.columns.tolist())
print(df.shape)

   year        municipality      sex     concept   value
0  2021  Abella de la Conca    males  population    89.0
1  2021  Abella de la Conca  females  population    82.0
2  2021  Abella de la Conca    total  population   171.0
3  2021              Abrera    males  population  6373.0
4  2021              Abrera  females  population  6247.0
['year', 'municipality', 'sex', 'concept', 'value']
(14220, 5)


Parámetros:

- `_LAST_`, en este caso, es para coger un número de años desde el más reciente.
    - Si cada año queremos descargar el dataset entero, será necesario ajustar `_LAST_` (incrementar en uno)
    - No usar `_LAST_` puede bloquear una consulta por superar el límite de filas. Esta es la forma mácil de filtrar.

- `lang` permite tener los datos direcamente en catalán, castellano o inglés. Por defecto es catalán.

In [15]:
url = "https://api.idescat.cat/taules/v2/pmh/446/477/mun/data"

response = requests.get(
    url,
    params={"_LAST_": 1}
)

response.raise_for_status()

dataset = pyjstat.Dataset.read(response.text)
df = dataset.write("dataframe")

print(df.head())
print(df.columns.tolist())
print(df.shape)

    any            municipi   sexe  concepte   value
0  2025  Abella de la Conca  homes  població    81.0
1  2025  Abella de la Conca  dones  població    75.0
2  2025  Abella de la Conca  total  població   156.0
3  2025              Abrera  homes  població  6639.0
4  2025              Abrera  dones  població  6588.0
['any', 'municipi', 'sexe', 'concepte', 'value']
(2844, 5)
